In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../")
import pandas as pd
from utils import ca as cu, behave as bu, plot as pu, db as db
import numpy as np
from paths.config import M2PConfig
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.stats import ranksums, spearmanr, skewnorm
from scipy.ndimage import gaussian_filter1d
from scipy.stats import ranksums, wilcoxon
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA


In [179]:
COLOR_PENK = (0, 0, 1, 1)
COLOR_NONPENK = (1, 0, 0, 1)

LINEWIDTH_HIST = 3
LINEWIDTH_FIT = 3

RESPONSE_TYPE = 'deconv_norm' #'deconv_norm'
RESPONSE_TYPE_ABR = 'norm. deconv.'
# Guassian smooth in frames
RESPONSE_SIGMA = 3
RESAMPLE_FACTOR = 10

COL_ABS_ROT = bu.AHV_FILT_GRAD + "-ABS"
COL_HD_SIN = bu.HD_ABS_FILT + "-SIN"
COL_HEADING_EGO_SIN = bu.HEADING_EGO_ABS + "-COS"
COL_HEADING_EGO_COS = bu.HEADING_EGO_ABS + "-SIN"
COL_HEADING_ALLO_SIN = bu.HEADING_ALLO_ABS + "-COS"
COL_HEADING_ALLO_COS = bu.HEADING_ALLO_ABS + "-SIN"
COL_HD_COS = bu.HD_ABS_FILT + "-COS"
COL_IS_ROT = "IS-ROT"
COL_IS_TRANS = "IS-TRANS"
COL_IS_LOCO = "IS-LOCO"
COL_DIST_TRAV = "distance-travelled"

In [180]:
cfg = M2PConfig()
df_exps, df_roi, df_ca, df_behave, df_behave_ca = db.get_ca_behave_data(cfg)

df_behave_ca[COL_ABS_ROT] = df_behave_ca[bu.AHV_FILT_GRAD].abs()
df_behave_ca[COL_IS_ROT] = bu.get_ahv_indexes(df_behave_ca)
df_behave_ca[COL_IS_TRANS] = bu.get_trans_indexes(df_behave_ca)
df_behave_ca[COL_IS_LOCO] = bu.get_loco_indexes(df_behave_ca)

#df_behave_ca[RESPONSE_TYPE] = df_behave_ca[RESPONSE_TYPE] + 0.001

df_behave_ca[COL_HD_SIN] = np.sin(np.radians(df_behave_ca[bu.HD_ABS_FILT]))
df_behave_ca[COL_HD_COS] = np.cos(np.radians(df_behave_ca[bu.HD_ABS_FILT]))
df_behave_ca[COL_HEADING_EGO_SIN] = np.sin(np.radians(df_behave_ca[bu.HEADING_EGO_ABS]))
df_behave_ca[COL_HEADING_EGO_COS] = np.cos(np.radians(df_behave_ca[bu.HEADING_EGO_ABS]))
df_behave_ca[COL_HEADING_ALLO_SIN] = np.sin(np.radians(df_behave_ca[bu.HEADING_ALLO_ABS]))
df_behave_ca[COL_HEADING_ALLO_COS] = np.cos(np.radians(df_behave_ca[bu.HEADING_ALLO_ABS]))



In [181]:
grp_df_cell = df_behave_ca.groupby(['exp_id', 'roi_id', 'celltype'])


# Fit models and put in roi table

In [189]:
for (exp_id, roi_id, cell_type), group in grp_df_cell:
    
    group[COL_DIST_TRAV] = group[bu.DIST_FILT_CM].cumsum()
    
        
    models = [[bu.AHV_FILT_GRAD,
               bu.SPEED_FILT_GRAD,
               bu.LOCO_SPEED_FILT_GRAD,
               bu.LIGHT_ON]]
    
    # models = [["time"],
    #           [COL_IS_ROT,
    #            COL_IS_TRANS,
    #            COL_IS_LOCO,
    #            bu.LIGHT_ON],
    #           [bu.AHV_FILT_GRAD,
    #            bu.SPEED_FILT_GRAD,
    #            bu.LOCO_SPEED_FILT_GRAD,
    #            bu.LIGHT_ON],
    #           [bu.SPEED_FILT_GRAD, 
    #            bu.AHV_FILT_GRAD, 
    #            bu.LOCO_SPEED_FILT_GRAD,
    #            bu.LIGHT_ON, 
    #            COL_HD_SIN, 
    #            COL_HD_COS, 
    #            COL_DIST_TRAV,
    #            bu.HEAD_X_FILT_MM,
    #            bu.HEAD_Y_FILT_MM,
    #            COL_HEADING_EGO_SIN,
    #            COL_HEADING_EGO_COS,
    #            COL_HEADING_ALLO_SIN,
    #            COL_HEADING_ALLO_COS]]
    

    
    r2_list = []
    
    
    
    param_dist = {
        'n_estimators': np.arange(100, 1000, 100),
        'max_depth': np.arange(10, 110, 10),
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['auto', 'sqrt'],
        'bootstrap': [True, False]
    }
    # todo asking chatgpt how to do hyper tuning, keeps failing when defined grid search params
    
    

    for m in models:
        
        x_train, x_test, y_train, y_test = train_test_split(group[m], 
                                                            group[RESPONSE_TYPE],                                
                                                            test_size=0.2, 
                                                            random_state=42)
        
        # n_train = 12000
        # x_train = group.iloc[0:n_train][m]
        # x_test = group.iloc[n_train+1:][m]
        # y_train = group.iloc[0:n_train][RESPONSE_TYPE]
        # y_test = group.iloc[n_train+1:][RESPONSE_TYPE]
        
#         rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
        
#         rf.fit(x_train, y_train)
#         pred_y = rf.predict(x_test)
#         r2_rf = r2_score(y_test, pred_y)
#         r2_list.append(r2_rf)
        
#         # Perform 5-fold cross-validation
#         scores = cross_val_score(rf, group[m], group[RESPONSE_TYPE], cv=5, scoring='r2')
        
#         #print('Cross-Validation Scores: {}'.format(scores))
#         print('Mean CV Score: {:.2f}'.format(scores.mean()))
#         #print('Standard Deviation: {:.2f}'.format(scores.std()))

        # Try grid search.
        # Still sucks, for at least one cell. And is super slow on laptop.
        # define the random forest regressor
        rf_reg = RandomForestRegressor()

        # define the parameter grid
        param_grid = {
            'n_estimators': [100, 200, 300],
            'max_depth': [5, 10, 15],
            'min_samples_split': [2, 4, 6],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2']
        }

        # define the scoring metric
        scorer = 'r2'

        # perform grid search
        grid_search = GridSearchCV(estimator=rf_reg, param_grid=param_grid, scoring=scorer, cv=5, n_jobs=-1)
        grid_search.fit(x_train, y_train)

        # print the best hyperparameters and corresponding score
        print('Best Hyperparameters: ', grid_search.best_params_)
        print('Best R2 Score: ', grid_search.best_score_)

        # predict on the test set using the best model
        y_pred = grid_search.best_estimator_.predict(x_test)
        r2_rf = r2_score(y_test, pred_y)
        r2_list.append(r2_rf)
        print('Test R2 Score: ', r2_rf)


    print(r2_list)
    

    
    
 
    
#     roi_index = np.logical_and(df_roi["exp_id"] == exp_id, df_roi["roi_id"] == roi_id)

#     df_roi.loc[roi_index, COL_CORR_TRANS_RHO] = corr_trans.statistic
#     df_roi.loc[roi_index, COL_CORR_TRANS_P] = corr_trans.pvalue
#     df_roi.loc[roi_index, COL_CORR_ROT_RHO] = corr_rot.statistic
#     df_roi.loc[roi_index, COL_CORR_ROT_P] = corr_rot.pvalue
    
    
    
    
    

/Users/tristan/opt/miniconda3/envs/suite2p/lib/python3.8/site-packages/sklearn/ensemble/_forest.py:414: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(


Best Hyperparameters:  {'max_depth': 5, 'max_features': 'auto', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best R2 Score:  0.019372473610922113
Test R2 Score:  -0.08892946692564041
[-0.08892946692564041]


/Users/tristan/opt/miniconda3/envs/suite2p/lib/python3.8/site-packages/sklearn/ensemble/_forest.py:414: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/tristan/opt/miniconda3/envs/suite2p/lib/python3.8/site-packages/sklearn/ensemble/_forest.py:414: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/tristan/opt/miniconda3/envs/suite2p/lib/python3.8/site-packages/sklearn/ensemble/_forest.py:414: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_feat

Best Hyperparameters:  {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best R2 Score:  0.014506765934897126
Test R2 Score:  -0.09985596865289659
[-0.09985596865289659]


/Users/tristan/opt/miniconda3/envs/suite2p/lib/python3.8/site-packages/sklearn/ensemble/_forest.py:414: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/tristan/opt/miniconda3/envs/suite2p/lib/python3.8/site-packages/sklearn/ensemble/_forest.py:414: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/tristan/opt/miniconda3/envs/suite2p/lib/python3.8/site-packages/sklearn/ensemble/_forest.py:414: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_feat

KeyboardInterrupt: 

In [169]:
group

,exp_id,roi_id,frame_id,bad,time,dFonF0,dFonF0_clean,dFonF0_norm_smooth,deconv_norm,deconv_norm_clean,...,Angular-head-velocity-filt-grad-ABS,IS-ROT,IS-TRANS,Head-direction-absolute-filt-SIN,Head-direction-absolute-filt-COS,Heading-ego-absolute-COS,Heading-ego-absolute-SIN,Heading-allo-absolute-COS,Heading-allo-absolute-SIN,distance-travelled
288000,20211028_11_25_50_1115465,0,0,False,0.000000,0.068211,0.0,0.093701,0.000000,0.0,...,37.150252,True,True,0.345526,-0.938409,0.902979,-0.429685,-0.995831,0.091217,0.019043
288023,20211028_11_25_50_1115465,0,1,False,9.765625,0.348810,0.0,0.092673,0.173050,0.0,...,133.518633,True,True,0.268939,-0.963157,0.391352,-0.920241,-0.624422,0.781087,0.128220
288046,20211028_11_25_50_1115465,0,2,False,19.531250,0.176463,0.0,0.090462,0.000000,0.0,...,88.608312,True,True,-0.096703,-0.995313,0.561082,-0.827760,-0.478405,0.878139,0.262747
288069,20211028_11_25_50_1115465,0,3,False,29.296875,0.258305,0.0,0.086972,0.000000,0.0,...,19.034008,True,True,-0.017098,-0.999854,0.870412,0.492324,-0.878702,-0.477370,0.334355
288092,20211028_11_25_50_1115465,0,4,False,39.062500,0.231194,0.0,0.082389,0.000000,0.0,...,24.549849,True,True,-0.054393,-0.998520,-0.072679,0.997355,0.018322,-0.999832,0.447370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
701885,20211028_11_25_50_1115465,0,17995,False,175732.421875,-0.035622,-0.0,0.007469,0.000000,0.0,...,8.552898,True,False,0.999895,0.014484,0.349988,-0.936754,-0.931587,-0.363519,363.322492
701908,20211028_11_25_50_1115465,0,17996,False,175742.187500,-0.136467,-0.0,0.009545,0.000000,0.0,...,8.647133,True,False,0.999565,0.029490,0.335888,-0.941902,-0.931587,-0.363519,363.323124
701931,20211028_11_25_50_1115465,0,17997,False,175751.953125,0.108951,0.0,0.011334,0.027208,0.0,...,8.738930,True,False,0.999003,0.044653,0.321558,-0.946890,-0.931587,-0.363519,363.323755
701954,20211028_11_25_50_1115465,0,17998,False,175761.718750,0.021077,0.0,0.012606,0.000000,0.0,...,8.832425,True,False,0.998154,0.060733,0.306269,-0.951945,-0.931587,-0.363519,363.324386


# Model whole data set lol

In [108]:
# Leave out distance travelled because can't be bothered doing the loop.

# m = [bu.SPEED_FILT_GRAD, 
#                bu.AHV_FILT_GRAD, 
#                bu.LIGHT_ON, 
#                COL_HD_SIN, 
#                COL_HD_COS, 
#                bu.HEAD_X_FILT_MM,
#                bu.HEAD_Y_FILT_MM,
#                bu.HEADING_EGO_ABS_FILT,
#                COL_HEADING_ALLO_SIN,
#                COL_HEADING_ALLO_COS]

# x_train, x_test, y_train, y_test = train_test_split(df_behave_ca[m], 
#                                                     df_behave_ca[RESPONSE_TYPE], 
#                                                     test_size=0.2, 
#                                                     random_state=42)

# rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)

# rf_regressor.fit(x_train, y_train)

# pred_y = rf_regressor.predict(x_test)

# r2_rf = r2_score(y_test, pred_y)

# print(r2_rf)

### Tell the model if it's a penk cell or not
df_behave_ca["is_penk"] = df_behave_ca["celltype"] == "penk" 

m = [bu.SPEED_FILT_GRAD, 
     bu.AHV_FILT_GRAD, 
     bu.LIGHT_ON, 
     COL_HD_SIN, 
     COL_HD_COS, 
     bu.HEAD_X_FILT_MM,
     bu.HEAD_Y_FILT_MM,
     bu.HEADING_EGO_ABS_FILT,
     COL_HEADING_ALLO_SIN,
     COL_HEADING_ALLO_COS,
     "is_penk"]

x_train, x_test, y_train, y_test = train_test_split(df_behave_ca[m], 
                                                    df_behave_ca[RESPONSE_TYPE], 
                                                    test_size=0.2, 
                                                    random_state=42)

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)

rf_regressor.fit(x_train, y_train)

pred_y = rf_regressor.predict(x_test)

r2_rf = r2_score(y_test, pred_y)

print(r2_rf)

-0.06730746636702967


In [106]:
print(r2_rf)

-0.06728174017537203


In [71]:
x_train

,Speed-filt-grad,Angular-head-velocity-filt-grad
30746,13.728898,300.220119
91722,0.477424,12.553728
39290,23.026883,45.547173
194106,0.026949,0.670669
265802,0.093440,-4.368877
...,...,...
180554,0.021459,1.356729
191434,0.181285,2.300277
86250,5.363638,-83.090495
13770,5.576194,-20.128173


In [ ]:
# old GLM stuff

    form_trans = "{resp} ~ Q('{trans}')".format(
                  resp=RESPONSE_TYPE, 
                  trans=bu.SPEED_FILT_GRAD)
    form_trans_rot = """{resp} ~ Q('{trans}') + 
                                 Q('{rot}')""".format(
                                 resp=RESPONSE_TYPE, 
                                 trans=bu.SPEED_FILT_GRAD, 
                                 rot=COL_ABS_ROT)
    form_trans_rot_x = """{resp} ~ Q('{trans}') + 
                                   Q('{rot}') + 
                                   Q('{trans}') * Q('{rot}')""".format(
                                  resp=RESPONSE_TYPE, 
                                  trans=bu.SPEED_FILT_GRAD, 
                                  rot=COL_ABS_ROT)
    form_trans_rot_x_light = """{resp} ~ Q('{trans}') + 
                                         Q('{rot}') + 
                                         Q('{trans}') * 
                                         Q('{rot}') + 
                                         Q('{light}')""".format(
                                         resp=RESPONSE_TYPE, 
                                         trans=bu.SPEED_FILT_GRAD, 
                                         rot=COL_ABS_ROT, 
                                         light=bu.LIGHT_ON)
    form_trans_rot_x_light_x = """{resp} ~ 
                               Q('{trans}') + 
                               Q('{rot}') + 
                               Q('{trans}') * Q('{light}') + 
                               Q('{trans}') * Q('{light}') + 
                               Q('{rot}') * Q('{light}') + 
                               Q('{trans}') * Q('{rot}') * Q('{light}')""".format(
                               resp=RESPONSE_TYPE, 
                               trans=bu.SPEED_FILT_GRAD,
                               rot=COL_ABS_ROT, 
                               light=bu.LIGHT_ON)
    
    form_trans_rot_x_light_x = """{resp} ~ 
                               Q('{trans}') + 
                               Q('{rot}') + 
                               Q('{trans}') * Q('{light}') + 
                               Q('{trans}') * Q('{light}') + 
                               Q('{rot}') * Q('{light}') + 
                               Q('{trans}') * Q('{rot}') * Q('{light}') + 
                               Q('{time}')""".format(
                               resp=RESPONSE_TYPE, 
                               trans=bu.SPEED_FILT_GRAD,
                               rot=COL_ABS_ROT, 
                               light=bu.LIGHT_ON,
                               time="time")
    
    # form_trans = "{resp} ~ Q('{trans}')".format(
    #               resp=RESPONSE_TYPE, 
    #               trans=COL_IS_TRANS)
    # form_trans_rot = """{resp} ~ Q('{trans}') + 
    #                              Q('{rot}')""".format(
    #                              resp=RESPONSE_TYPE, 
    #                              trans=COL_IS_TRANS, 
    #                              rot=COL_IS_ROT)
    # form_trans_rot_x = """{resp} ~ Q('{trans}') + 
    #                                Q('{rot}') + 
    #                                Q('{trans}')) * Q('{rot}')""".format(
    #                               resp=RESPONSE_TYPE, 
    #                               trans=COL_IS_TRANS, 
    #                               rot=COL_IS_ROT)
    # form_trans_rot_x_light = """{resp} ~ Q('{trans}') + 
    #                                      Q('{rot}')_ + 
    #                                      Q('{trans}') * 
    #                                      Q('{rot}') + 
    #                                      Q('{light}')""".format(
    #                                      resp=RESPONSE_TYPE, 
    #                                      trans=COL_IS_TRANS, 
    #                                      rot=COL_IS_ROT, 
    #                                      light=bu.LIGHT_ON)
    # form_trans_rot_x_light_x = """{resp} ~ 
    #                            Q('{trans}') + 
    #                            Q('{rot}')_ + 
    #                            Q('{trans}') * 
    #                            Q('{rot}') + 
    #                            Q('{light}')
    #                            Q('{trans}') * Q('{light}') + 
    #                            Q('{rot}')) * Q('{light}') + 
    #                            Q('{trans}')) * Q('{rot}') * Q('{light}')""".format(
    #                            resp=RESPONSE_TYPE, 
    #                            trans=COL_IS_TRANS,
    #                            rot=COL_IS_ROT, 
    #                            light=bu.LIGHT_ON)
    
#     fam = sm.families.Gaussian(sm.families.links.identity())
#     #fam = sm.families.Gamma(sm.families.links.log())
#     model_lg_trans = smf.glm(form_trans, data=group, family=fam)
#    model_lg_trans_rot = smf.glm(form_trans_rot, data=group, family=fam)
#     model_lg_trans_rot_x = smf.glm(form_trans_rot_x, data=group, family=fam)
#     model_lg_trans_rot_x_light = smf.glm(form_trans_rot_x_light, data=group, family=fam)
#    model_lg_trans_rot_x_light_x = smf.glm(form_trans_rot_x_light_x, data=group, family=fam)
    
#     results_lg_trans = model_lg_trans.fit()
#    results_lg_trans_rot = model_lg_trans_rot.fit()
#     results_lg_trans_rot_x = model_lg_trans_rot_x.fit()
#     results_lg_trans_rot_x_light = model_lg_trans_rot_x_light.fit()
#    results_lg_trans_rot_x_light_x = model_lg_trans_rot_x_light_x.fit()
    
    #pred_lg = results_lg_trans_rot_x_light_x.predict(group[[bu.SPEED_FILT_GRAD, COL_ABS_ROT, bu.LIGHT_ON, "time"]])
    #r2_lg = r2_score(group[RESPONSE_TYPE], pred_lg)

In [ ]:

penk_indexes = df_roi["celltype"] == "penk"
nonpenk_indexes = ~penk_indexes
    
# Plot the main response against the null response.
plt.figure()
plt.scatter(df_roi.loc[penk_indexes][COL_CORR_TRANS_RHO],
            df_roi.loc[penk_indexes][COL_CORR_ROT_RHO],
            color=COLOR_PENK, label="penk")
plt.scatter(df_roi.loc[nonpenk_indexes][COL_CORR_TRANS_RHO],
            df_roi.loc[nonpenk_indexes][COL_CORR_ROT_RHO],
            color=COLOR_NONPENK, label="nonpenk")
pu.square_plot()
plt.xlabel("Translation ρ")
plt.ylabel("Rotation ρ")
plt.show()
plt.close()

result = wilcoxon(df_roi.loc[penk_indexes][COL_CORR_TRANS_RHO], 
                  df_roi.loc[penk_indexes][COL_CORR_ROT_RHO])
print(result)
result = wilcoxon(df_roi.loc[nonpenk_indexes][COL_CORR_TRANS_RHO], 
                  df_roi.loc[nonpenk_indexes][COL_CORR_ROT_RHO])
print(result)

# Calculate and index and plot a histgram
df_roi[COL_CORR_TRANS_ROT_DIFF_RHO] = df_roi[COL_CORR_TRANS_RHO] - df_roi[COL_CORR_ROT_RHO]

# Translation
plt.figure()
hist_data = df_roi.loc[penk_indexes][COL_CORR_TRANS_RHO]
plt.hist(hist_data, 
         color=COLOR_PENK, label="penk",
         weights=np.ones(len(hist_data)) / len(hist_data))
hist_data = df_roi.loc[nonpenk_indexes][COL_CORR_TRANS_RHO]
plt.hist(hist_data, color=COLOR_NONPENK, 
         label="nonpenk",
         weights=np.ones(len(hist_data)) / len(hist_data),
         histtype=u'step',
         linewidth=LINEWIDTH_HIST)
plt.xlabel("Translation ρ")
plt.ylabel("Fraction of cells")

# Rotation
plt.figure()
hist_data = df_roi.loc[penk_indexes][COL_CORR_ROT_RHO]
plt.hist(hist_data, 
         color=COLOR_PENK, label="penk",
         weights=np.ones(len(hist_data)) / len(hist_data))
hist_data = df_roi.loc[nonpenk_indexes][COL_CORR_ROT_RHO]
plt.hist(hist_data, color=COLOR_NONPENK, 
         label="nonpenk",
         weights=np.ones(len(hist_data)) / len(hist_data),
         histtype=u'step',
         linewidth=LINEWIDTH_HIST)
plt.xlabel("Rotation ρ")
plt.ylabel("Fraction of cells")

# Translation - rotation
plt.figure()
hist_data = df_roi.loc[penk_indexes][COL_CORR_TRANS_ROT_DIFF_RHO]
plt.hist(hist_data, 
         color=COLOR_PENK, label="penk",
         weights=np.ones(len(hist_data)) / len(hist_data))
hist_data = df_roi.loc[nonpenk_indexes][COL_CORR_TRANS_ROT_DIFF_RHO]
plt.hist(hist_data, color=COLOR_NONPENK, 
         label="nonpenk",
         weights=np.ones(len(hist_data)) / len(hist_data),
         histtype=u'step',
         linewidth=LINEWIDTH_HIST)
plt.xlabel("Trans. - rot. ρ")
plt.ylabel("Fraction of cells")